ANALISIS CON BALANCE DE PESOS PARA RANDOM FOREST


1.Dataset completo

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split

df_Comp_pesos = pd.read_csv("/content/dataset1_completo_pesos.csv")

df_1 = df_Comp_pesos.drop(columns=['mental_health_index', 'dropout_risk'])

X = df_1.drop(columns=["burnout_score", "peso"])
y = df_1["burnout_score"]
pesos = df_1["peso"]

In [ ]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split

t_inicio = time.time()

# =============================================================================
# 1. DIVISIÓN DEL CONJUNTO DE DATOS (CON PESOS)
# =============================================================================
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_encoded, y, pesos, test_size=0.30, random_state=42, stratify=y
)

# =============================================================================
# 2. CONFIGURACIÓN ACOTADA PARA 1M DE FILAS (RANDOM FOREST + PESOS)
# =============================================================================
param_grid = {
    'n_estimators': [80, 120, 150],       # Acotado para no disparar el tiempo en 1M de filas
    'max_depth': [6, 10, 14],             # Evita árboles gigantes que tardan mucho
    'min_samples_split': [5, 10],         # Divisiones más estables
    'min_samples_leaf': [2, 5],           # Hojas mínimas para reducir coste y sobreajuste
    'max_features': ['sqrt', 'log2'],     # Menos variables evaluadas por corte = más rápido
    'bootstrap': [True]
}

print("=== Iniciando Búsqueda Rápida de Hiperparámetros (Random Forest + Pesos) ===")

rf_base = RandomForestClassifier(
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=5,            # 5 combinaciones x 2 folds = 10 ajustes (prueba rápida)
    scoring='accuracy',  # Métrica guía en validación cruzada
    cv=2,                # 2 pliegues para calibrar tiempo rápido en Colab
    random_state=42,
    n_jobs=-1
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA PONDERADA POR PESOS
# =============================================================================
random_search.fit(X_train, y_train, sample_weight=pesos_train)
print(f"Búsqueda terminada en {(time.time() - t_inicio) / 60:.2f} minutos.")

# Extraemos los índices de los 3 mejores modelos según el ranking en CV
resultados_cv = pd.DataFrame(random_search.cv_results_)
top_3_indices = resultados_cv.sort_values(by='rank_test_score').index[:3]

print("\nTOP 3 MEJORES MODELOS ENCONTRADOS:\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS PONDERADAS
# =============================================================================
for rank, idx in enumerate(top_3_indices, start=1):
    params_top = resultados_cv.loc[idx, 'params']
    cv_score_top = resultados_cv.loc[idx, 'mean_test_score']

    modelo_top = RandomForestClassifier(random_state=42, n_jobs=-1, **params_top)
    modelo_top.fit(X_train, y_train, sample_weight=pesos_train)

    y_pred_train_top = modelo_top.predict(X_train)
    y_pred_test_top = modelo_top.predict(X_test)

    # TRAIN (ponderado por pesos_train)
    accuracy_train = accuracy_score(y_train, y_pred_train_top, sample_weight=pesos_train)
    precision_train = precision_score(y_train, y_pred_train_top, sample_weight=pesos_train)
    recall_train = recall_score(y_train, y_pred_train_top, sample_weight=pesos_train)
    f1_train = f1_score(y_train, y_pred_train_top, sample_weight=pesos_train)

    # TEST (ponderado por pesos_test)
    accuracy_test = accuracy_score(y_test, y_pred_test_top, sample_weight=pesos_test)
    precision_test = precision_score(y_test, y_pred_test_top, sample_weight=pesos_test)
    recall_test = recall_score(y_test, y_pred_test_top, sample_weight=pesos_test)
    f1_test = f1_score(y_test, y_pred_test_top, sample_weight=pesos_test)

    # =========================================================================
    # 5. PROYECCIÓN DE RESULTADOS
    # =========================================================================
    print(f"\n--- TOP {rank} | CV Score: {cv_score_top:.4f} ---")
    print(f"Hiperparámetros: {params_top}")
    print("Resultados del Modelo RANDOM FOREST (PESOS)")
    print(f"Métrica      |      TRAIN     |     TEST   ")
    print(f"-------------|----------------|---------------")
    print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
    print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
    print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
    print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Rápida de Hiperparámetros (Random Forest + Pesos) ===
Búsqueda terminada en 39.40 minutos.

TOP 3 MEJORES MODELOS ENCONTRADOS:


--- TOP 1 | CV Score: 0.8957 ---
Hiperparámetros: {'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 14, 'bootstrap': True}
Resultados del Modelo RANDOM FOREST (PESOS)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.9219     |    0.8655
Precisión    |     0.9071     |    0.8873
Sensibilidad |     0.9400     |    0.8374
F1-Score     |     0.9233     |    0.8616

--- TOP 2 | CV Score: 0.8636 ---
Hiperparámetros: {'n_estimators': 120, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}
Resultados del Modelo RANDOM FOREST (PESOS)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8841     |    0.8731
Precisió

2.Dataset sin features de baja correlacionc on el target

In [ ]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split

df_3 = pd.read_csv("/content/dataset3_reducido_pesos.csv")

X = df_3.drop(columns=["burnout_score", "peso"])
y = df_3["burnout_score"]
pesos = df_3["peso"]

In [ ]:
t_inicio = time.time()

# =============================================================================
# 1. DIVISIÓN DEL CONJUNTO DE DATOS (CON PESOS)
# =============================================================================
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X_encoded, y, pesos, test_size=0.30, random_state=42, stratify=y
)

# =============================================================================
# 2. CONFIGURACIÓN ACOTADA PARA 1M DE FILAS (RANDOM FOREST + PESOS)
# =============================================================================
param_grid = {
    'n_estimators': [80, 120, 150],       # Acotado para no disparar el tiempo en 1M de filas
    'max_depth': [6, 10, 14],             # Evita árboles gigantes que tardan mucho
    'min_samples_split': [5, 10],         # Divisiones más estables
    'min_samples_leaf': [2, 5],           # Hojas mínimas para reducir coste y sobreajuste
    'max_features': ['sqrt', 'log2'],     # Menos variables evaluadas por corte = más rápido
    'bootstrap': [True]
}

print("=== Iniciando Búsqueda Rápida de Hiperparámetros (Random Forest + Pesos) ===")

rf_base = RandomForestClassifier(
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=5,            # 5 combinaciones x 2 folds = 10 ajustes (prueba rápida)
    scoring='accuracy',  # Métrica guía en validación cruzada
    cv=2,                # 2 pliegues para calibrar tiempo rápido en Colab
    random_state=42,
    n_jobs=-1
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA PONDERADA POR PESOS
# =============================================================================
random_search.fit(X_train, y_train, sample_weight=pesos_train)
print(f"Búsqueda terminada en {(time.time() - t_inicio) / 60:.2f} minutos.")

# Extraemos los índices de los 3 mejores modelos según el ranking en CV
resultados_cv = pd.DataFrame(random_search.cv_results_)
top_3_indices = resultados_cv.sort_values(by='rank_test_score').index[:3]

print("\nTOP 3 MEJORES MODELOS ENCONTRADOS:\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS PONDERADAS
# =============================================================================
for rank, idx in enumerate(top_3_indices, start=1):
    params_top = resultados_cv.loc[idx, 'params']
    cv_score_top = resultados_cv.loc[idx, 'mean_test_score']

    modelo_top = RandomForestClassifier(random_state=42, n_jobs=-1, **params_top)
    modelo_top.fit(X_train, y_train, sample_weight=pesos_train)

    y_pred_train_top = modelo_top.predict(X_train)
    y_pred_test_top = modelo_top.predict(X_test)

    # TRAIN (ponderado por pesos_train)
    accuracy_train = accuracy_score(y_train, y_pred_train_top, sample_weight=pesos_train)
    precision_train = precision_score(y_train, y_pred_train_top, sample_weight=pesos_train)
    recall_train = recall_score(y_train, y_pred_train_top, sample_weight=pesos_train)
    f1_train = f1_score(y_train, y_pred_train_top, sample_weight=pesos_train)

    # TEST (ponderado por pesos_test)
    accuracy_test = accuracy_score(y_test, y_pred_test_top, sample_weight=pesos_test)
    precision_test = precision_score(y_test, y_pred_test_top, sample_weight=pesos_test)
    recall_test = recall_score(y_test, y_pred_test_top, sample_weight=pesos_test)
    f1_test = f1_score(y_test, y_pred_test_top, sample_weight=pesos_test)

    # =========================================================================
    # 5. PROYECCIÓN DE RESULTADOS
    # =========================================================================
    print(f"\n--- TOP {rank} | CV Score: {cv_score_top:.4f} ---")
    print(f"Hiperparámetros: {params_top}")
    print("Resultados del Modelo RANDOM FOREST (PESOS)")
    print(f"Métrica      |      TRAIN     |     TEST   ")
    print(f"-------------|----------------|---------------")
    print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
    print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
    print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
    print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Rápida de Hiperparámetros (Random Forest + Pesos) ===
Búsqueda terminada en 39.83 minutos.

TOP 3 MEJORES MODELOS ENCONTRADOS:


--- TOP 1 | CV Score: 0.8965 ---
Hiperparámetros: {'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 14, 'bootstrap': True}
Resultados del Modelo RANDOM FOREST (PESOS)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.9224     |    0.8650
Precisión    |     0.9085     |    0.8883
Sensibilidad |     0.9395     |    0.8350
F1-Score     |     0.9237     |    0.8608

--- TOP 2 | CV Score: 0.8662 ---
Hiperparámetros: {'n_estimators': 120, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}
Resultados del Modelo RANDOM FOREST (PESOS)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.8856     |    0.8747
Precisió